# R / Tidyverse Smoke Test

Verifies the DoML R environment and tidyverse stack inside the container.
Run via: `docker compose run --rm jupyter papermill notebooks/test_r_tidyverse.ipynb notebooks/test_r_tidyverse.ipynb -k ir`

In [1]:
set.seed(42)
library(dplyr)
library(ggplot2)
library(tidyr)
library(readr)
library(purrr)
library(tibble)
cat("tidyverse core libraries loaded\n")
cat(sprintf("dplyr version: %s\n", packageVersion("dplyr")))
cat(sprintf("ggplot2 version: %s\n", packageVersion("ggplot2")))


Attaching package: ‘dplyr’




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




tidyverse core libraries loaded


dplyr version: 1.2.0


ggplot2 version: 4.0.2


In [2]:
library(tidymodels)
cat(sprintf("tidymodels version: %s\n", packageVersion("tidymodels")))

── Attaching packages ────────────────────────────────────── tidymodels 1.4.1 ──



✔ broom        1.0.12     ✔ rsample      1.3.2 
✔ dials        1.4.2      ✔ tailor       0.1.0 
✔ infer        1.1.0      ✔ tune         2.0.1 
✔ modeldata    1.5.1      ✔ workflows    1.3.0 
✔ parsnip      1.4.1      ✔ workflowsets 1.1.1 
✔ recipes      1.3.2      ✔ yardstick    1.3.2 



── Conflicts ───────────────────────────────────────── tidymodels_conflicts() ──
✖ scales::discard() masks purrr::discard()
✖ dplyr::filter()   masks stats::filter()
✖ dplyr::lag()      masks stats::lag()
✖ yardstick::spec() masks readr::spec()
✖ recipes::step()   masks stats::step()



tidymodels version: 1.4.1


In [3]:
df <- tibble(
  x = rnorm(100, mean = 5, sd = 2),
  y = rnorm(100, mean = 10, sd = 3),
  group = sample(c("A", "B"), 100, replace = TRUE)
)

result <- df %>%
  group_by(group) %>%
  summarise(
    n = n(),
    mean_x = mean(x),
    mean_y = mean(y),
    .groups = "drop"
  )

print(result)
cat("PASS: R/tidyverse stack operational\n")

# A tibble: 2 × 4
  group     n mean_x mean_y
  <chr> <int>  <dbl>  <dbl>
1 A        45   4.87   9.40
2 B        55   5.22  10.0 


PASS: R/tidyverse stack operational


In [4]:
data_split <- initial_split(df, prop = 0.8)
train_data <- training(data_split)
test_data  <- testing(data_split)

rec <- recipe(y ~ x + group, data = train_data) %>%
  step_dummy(group)

lm_spec <- linear_reg() %>% set_engine("lm")

wf <- workflow() %>%
  add_recipe(rec) %>%
  add_model(lm_spec) %>%
  fit(data = train_data)

preds <- predict(wf, test_data)
cat(sprintf("tidymodels linear_reg fit: %d predictions generated\n", nrow(preds)))
cat("PASS: tidymodels operational\n")

tidymodels linear_reg fit: 20 predictions generated


PASS: tidymodels operational
